# Exercise 4: Delta Table Creation and Perform of a MERGE instruction

The objective of this exercise is to perform a `MERGE INTO` on a existing Delta Table, using a DataFrame with changes.

## Step 0: Delta Table creation 

We will begin with a spark DataFrame built from a list of tuples, for this exercise purpose we will limit to use only 2 columns `("id", "value")`. This DataFrame will be used to create a Delta table.

It is important to clarify that the command `df.write.format("delta")` creates the Delta Files, in other words, it writes the `Parquet files` + `_delta_log` in the specified folder in the `.save(<path>)` option. This is by definition a "physical" Delta table, the thing is that it is not registered yet in the Unity Catalog. It can be read using Spark, but will not appear in the Catalog.

On the other hand the command enclosed in the `spark.sql()` instruction does not write data on the table, but, it registers it in the Catalog giving it an official name, thus being able to be queried from the Unity Catalog via SQL.


| Layer |	What is it? | What is for? |
|----------|----------|-------------|
| **Delta files** |	Archivos Parquet + delta_log |	Physical Persistency |
| **Unity Catalog table** |	Entrada en el catálogo | SQL, permissions, lineage, gobernance |

However it must be noticed that this approach only works for a `Premium/Enterprise` edition of Databricks since these editions allow to register tables using locations such as `s3://`, `abfss://`, `gs://`. In these versions Unity Catalog allows to register external tables.

If you are using a Free Edition is enough to use the `saveAsTable("catalog.schema.table")` inside `df.write()` command. In this case the Delta Files are created in an administrated volume, the table is registered in automatic and the most important part is that it does not require a LOCATION value.

In the praxis and broadly speaking we sould rather use `saveAsTable("catalog.schema.table")` for internal Lakehouse tables, fully internal pipelines inside Databricks, and would rather prefer to use `CREATE TABLE ... LOCATION` for external tables stored in `(S3, ADLS, GCS)` making Unity Catalog only store metadata. 

In [0]:
# 1. Tabla base
data = [(1, "A"), (2, "B")]
df = spark.createDataFrame(data, ["id", "value"])

#df.write.format("delta") \
#    .mode("overwrite") \
#    .option("overwriteSchema", "true") \
#    .save("/Volumes/workspace/default/streaming_demo/merge_demo")

#spark.sql("""
#CREATE TABLE IF NOT EXISTS workspace.default.merge_demo
#USING DELTA
#LOCATION '/Volumes/workspace/default/streaming_demo/merge_demo'
#""")

# 2. Crear tabla Delta + registrar en UC automáticamente
df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.merge_demo")

# Step 1: Updates DataFrame Creation

Once we created the delta table, we will create a Spark DataFrame and store it as a View 

In [0]:
# 3. DataFrame de updates
updates = [(1, "A1"), (3, "C")]
df_upd = spark.createDataFrame(updates, ["id", "value"])
df_upd.createOrReplaceTempView("updates")

# Step 2: MERGE INTO

We perform the `MERGE INTO` instruction within `spark.sql()` command. This instruction only requires that the Delta Table exists in the Unity Catalog.

This command does not create new tables, it updates or inserts in a table that already exists.

In [0]:
# 4. MERGE INTO
spark.sql("""
  MERGE INTO workspace.default.merge_demo AS t
  USING updates AS u
  ON t.id = u.id
  WHEN MATCHED THEN UPDATE SET value = u.value
  WHEN NOT MATCHED THEN INSERT *
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

# Step 3: Validation

Since `MERGE INTO` does not create new tables it will only modify the one that we specified, therefore is enough to check on `workspace.default.merge_demo` to validate the exercise.

In [0]:
%sql
SELECT * FROM workspace.default.merge_demo;

id,value
2,B
1,A1
3,C
